# 02 — Prepare the recording list, step by step

**Goal:** make a table describing our Paderborn recordings and check which ones
can be used provisionally for analysis.

Follow one step at a time: **find files → describe files → check recordings → save results**.
This notebook does not filter, resample, create windows or train a model.

**Before starting:** select your project `.venv` kernel and change `project_root`
in Step 2 if needed. Run from the beginning after restarting the kernel.
The last steps update three CSVs under your project's `metadata` folder.
Raw `.mat` files are never changed.

Code cells are intentionally short. Two small functions are used only to avoid
repeating the same file-reading and checking steps for every recording.
`assert` means: stop here if the stated condition is not true.


## Step 1: Import the tools

**What:** import the tools for paths, arrays, tables, MATLAB files and display.
**Why:** the remaining cells use these names. No data is loaded yet.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.io import loadmat
from IPython.display import display


## Step 2: Set the folders and nominal sampling rate

**What:** tell Python where the project is stored.
**Why:** one root path keeps file locations consistent.
`fs` is the documented nominal rate; assigning it does not change the samples.
Use it provisionally only after the checks below. It is not inferred from array length.


In [ ]:
project_root = Path(r"C:\Users\redhu\OneDrive\Desktop\Bearing_fault_detection")
raw_folder = project_root / "data" / "raw" / "paderborn"
metadata_folder = project_root / "metadata"
fs = 64_000
assert raw_folder.is_dir(), "Check the raw-data folder path."


## Step 3: Find the MATLAB files

**What:** find `.mat` files, including those in nested folders.
**Why:** build a fresh list from disk instead of reloading an old inventory.
`rglob` searches subfolders; `sorted` puts paths in predictable text order.
The previous dataset contained 560 files. The result below is calculated afresh.


In [ ]:
files = sorted(raw_folder.rglob("*.mat"))
assert files, "No MATLAB files were found."
print("Files found:", len(files))
print("First file:", files[0].name)


## Step 4: Understand one filename

**What:** split a filename at its underscores. `.stem` removes `.mat`.
**Why:** each part supplies one item of recording metadata.

| Position | Example | Meaning |
|---|---|---|
| 0 | N09 | Speed-setting code |
| 1 | M07 | Torque-setting code |
| 2 | F10 | Force-setting code |
| 3 | K001 | Physical bearing |
| 4 | 1 | Recording repetition |

Filename codes describe operating settings, not measurements extracted from channels.


In [ ]:
parts = files[0].stem.split("_")
print(parts)


## Step 5: Make one table row for each file

**What:** repeat the filename split and store one dictionary per file.
**Why:** a registry needs one row per recording, not one row per vibration sample.
`int` makes repetitions numerical; `relative_to` stores a portable file location.
The filename pattern is checked before using its five parts.


In [ ]:
records = []
for path in files:
    parts = path.stem.split("_")
    assert len(parts) == 5 and parts[4].isdigit(), f"Check filename: {path.name}"
    assert int(parts[4]) > 0, f"Check repetition number: {path.name}"
    records.append({
        "recording_id": path.stem,
        "bearing_id": parts[3],
        "recording_no": int(parts[4]),
        "speed_code": parts[0],
        "torq_code": parts[1],
        "force_code": parts[2],
        "relative_path": path.relative_to(raw_folder).as_posix()
    })


## Step 6: Turn the rows into a table

**What:** create a pandas DataFrame and sort its rows.
**Why:** numerical repetition order is 1, 2, 3, rather than 1, 10, 11.
`reset_index` supplies a new row index after sorting; it does not change IDs.


In [ ]:
registry = pd.DataFrame(records)
order = ["bearing_id", "speed_code", "torq_code", "force_code", "recording_no"]
registry = registry.sort_values(order).reset_index(drop=True)
display(registry.head())


## Step 7: Assign the known bearing conditions

**What:** use the selected dataset's bearing labels.
**Why:** labels must come from dataset metadata, not how a waveform looks.
`map` looks up each bearing in the dictionary. Unrecognised bearings produce
missing values, which the validation step will reject.


In [ ]:
conditions = {
    "K001": "healthy", "K002": "healthy", "K003": "healthy",
    "KA04": "faulty", "KA15": "faulty", "KI04": "faulty", "KI14": "faulty"
}
registry["condition"] = registry["bearing_id"].map(conditions)


## Step 8: Assign development and final-test bearings

**What:** attach the agreed partition to every recording.
**Why:** all repetitions and later windows of one physical bearing must remain
in the same partition. Final-test data stays outside diagnostic exploration.


In [ ]:
partitions = {
    "K001": "development", "K002": "development",
    "KA04": "development", "KI04": "development",
    "K003": "final_test", "KA15": "final_test", "KI14": "final_test"
}
registry["partition"] = registry["bearing_id"].map(partitions)


## Step 9: Check the inventory

**What:** check the planned bearings, unique IDs, complete metadata and partition grouping.
**Why:** wrong labels or duplicate IDs would make later joins and evaluations unreliable.
The bearing-set check assumes our seven-bearing project scope. Review the plan
explicitly before adding or removing bearings.


In [ ]:
assert set(registry["bearing_id"]) == set(conditions), "Check the seven-bearing dataset scope."
assert not registry["recording_id"].duplicated().any(), "Duplicate recording IDs."
assert not registry.isna().any().any(), "Missing metadata or unknown bearing."
assert registry.groupby("bearing_id")["partition"].nunique().eq(1).all()
print(registry["bearing_id"].value_counts().sort_index())
print(registry.groupby(["partition", "condition"]).size())


## Step 10: Add readable operating settings

**What:** decode the setting codes and record measurement assumptions.
**Why:** speed, torque and force matter when comparing vibration.
These are documented settings, not measured values. The amplitude unit remains
unverified; an empty unit field does not mean g or m/s².


In [ ]:
registry["speed_setting_rpm"] = registry["speed_code"].map({"N09": 900, "N15": 1500})
registry["torque_setting_nm"] = registry["torq_code"].map({"M01": 0.1, "M07": 0.7})
registry["radial_force_setting_n"] = registry["force_code"].map({"F04": 400, "F10": 1000})
registry["vibration_channel"] = "vibration_1"
registry["nominal_fs_hz"] = fs
registry["vibration_unit_status"] = "unverified"
assert not registry.isna().any().any(), "Unknown operating-setting code."


## Step 11: Read vibration and its matching time axis

**What:** define the file-reading steps once.
**Why:** the audit will use the same reader for every recording.

| Expression | Meaning |
|---|---|
| `loadmat(path)[path.stem][0, 0]` | Open this file's recording structure |
| `recording["Y"].ravel()` | Put its channels into a one-dimensional array |
| `names.index("vibration_1")` | Locate vibration by name |
| `XIndex - 1` | Convert MATLAB's axis reference to Python indexing |
| `Data.ravel().astype(float)` | Extract a one-dimensional numerical array |

The three returned results are vibration, timestamps and a raster-match flag.
The guards prevent selecting an ambiguous channel or an invalid time-axis index.
This reader assumes the inspected dataset structure; missing fields/variables
stop execution instead of silently skipping a file.


In [ ]:
def read_vibration(path):
    recording = loadmat(path)[path.stem][0, 0]
    channels = recording["Y"].ravel()
    names = [channel["Name"].item() for channel in channels]
    assert names.count("vibration_1") == 1, f"Check vibration channel: {path.name}"
    vibration = channels[names.index("vibration_1")]
    index = int(vibration["XIndex"].item()) - 1
    axes = recording["X"].ravel()
    assert 0 <= index < len(axes), f"Check XIndex: {path.name}"
    axis = axes[index]
    x = vibration["Data"].ravel().astype(float)
    time = axis["Data"].ravel().astype(float)
    rasters_match = vibration["Raster"].item() == axis["Raster"].item()
    return x, time, rasters_match


## Step 12: Try the reader on one development recording

**What:** get one development row, reconstruct its path and call the reader.
**Why:** understand the returned values before running the full audit.
The first row may need timing review; using it for structural inspection does not
make it analysis-eligible. Matching counts alone do not establish accurate timing.


In [ ]:
row = registry[registry["partition"] == "development"].iloc[0]
example_path = raw_folder / row["relative_path"]
x, time, rasters_match = read_vibration(example_path)
print(row["recording_id"], "— samples:", x.size, "— time values:", time.size)


## Step 13: Check timing on the example

**What:** calculate the time span and full-span rate estimate.
**Why:** N timestamps have N − 1 intervals. For example, timestamps 0, 0.25,
0.50, 0.75 and 1.00 contain five samples but four intervals: 4 / 1 = 4 Hz.
The endpoint-based estimate is a consistency check, not proof of the actual rate.
Guards stop this demonstration if its timestamps are unusable.


In [ ]:
assert time.size >= 2 and np.isfinite(time).all(), "Check example timestamps."
assert (np.diff(time) > 0).all(), "Example time must increase."
span = time[-1] - time[0]
print("Stored time span (s):", span)
print("Rate from span (Hz):", (time.size - 1) / span)
print("Documented nominal rate (Hz):", fs)


## Step 14: Define the checks for one recording

**What:** return one dictionary of timing and quality results.
**Why:** every file must receive the same checks.
This is the longest cell because it collects the results in one reusable place.

- `signal_finite`: nonempty vibration with no NaN or infinity.
- `time_finite`: at least two timestamps, all finite.
- `time_increasing`: finite timestamps that strictly increase.
- `rasters_match`: vibration and its selected axis have matching raster names.
- Invalid timestamps produce a missing (`NaN`) rate instead of an invented rate.

These checks do not establish calibration, absence of clipping or perfect
uniformity of individual sampling intervals.


In [ ]:
def inspect_recording(path):
    x, time, rasters_match = read_vibration(path)
    time_finite = time.size >= 2 and np.isfinite(time).all()
    time_increasing = time_finite and (np.diff(time) > 0).all()
    span = time[-1] - time[0] if time_finite else np.nan
    rate = (time.size - 1) / span if time_increasing and span > 0 else np.nan
    return {
        "recording_id": path.stem,
        "sample_count": x.size,
        "time_count": time.size,
        "axis_span_s": span,
        "axis_rate_hz": rate,
        "signal_finite": bool(x.size > 0 and np.isfinite(x).all()),
        "time_finite": bool(time_finite),
        "time_increasing": bool(time_increasing),
        "rasters_match": rasters_match
    }


## Step 15: Try the checks on the example

**What:** run the checker once and display each result.
**Why:** see the contents of one audit row before collecting hundreds of them.
`pd.Series` displays the dictionary as a labelled column.


In [ ]:
display(pd.Series(inspect_recording(example_path)))


## Step 16: Check all recordings

**What:** run the checker on each registry path and build an audit table.
**Why:** a timing issue in one file cannot be assumed to apply to every file.
Each file is read once in this full audit pass. Final-test files receive only
these structural/timing checks, not diagnostic exploration or feature tuning.


In [ ]:
audit_records = []
for relative_path in registry["relative_path"]:
    audit_records.append(inspect_recording(raw_folder / relative_path))
audit_table = pd.DataFrame(audit_records)
print("Recordings checked:", len(audit_table))


## Step 17: Measure disagreement from the nominal rate

**What:** calculate the absolute percentage difference from 64 kHz.
**Why:** compare every recording on the same scale, regardless of whether its
estimate is above or below nominal. Multiply the fraction by 100 for percent.


In [ ]:
audit_table["axis_rate_difference_pct"] = (
    (audit_table["axis_rate_hz"] - fs) / fs
).abs() * 100


## Step 18: Flag timing cases for review

**What:** flag missing/invalid estimates and differences above 0.1%.
**Why:** make the provisional screening decision explicit and repeatable.
The threshold is our project choice, not a manufacturer specification.
Passing the screen means agreement over the complete span, not verified timing.


In [ ]:
audit_table["needs_timing_review"] = (
    ~np.isfinite(audit_table["axis_rate_difference_pct"])
    | (audit_table["axis_rate_difference_pct"] > 0.1)
)
audit_table["timing_status"] = np.where(
    audit_table["needs_timing_review"], "needs_review", "within_span_tolerance"
)
print(audit_table["timing_status"].value_counts())


## Step 19: Summarize the quality results

**What:** count how many recordings pass each quality check and show the largest
finite timing disagreements.
**Why:** timing agreement and basic quality are separate questions.
In the previous real-data run all 560 passed these four quality checks, but
79 needed timing review. These are historical reference counts, not forced outputs.


In [ ]:
quality_columns = ["signal_finite", "time_finite", "time_increasing", "rasters_match"]
print(audit_table[quality_columns].sum())
print("Length mismatches:", (audit_table["sample_count"] != audit_table["time_count"]).sum())
display(audit_table.sort_values("axis_rate_difference_pct", ascending=False).head())


## Step 20: Attach the audit results to the registry

**What:** match audit rows to registry rows using `recording_id`.
**Why:** one complete row should describe each recording.
`how="left"` preserves the inventory; `validate="one_to_one"` rejects duplicate
join keys. A missing audit row must stop the notebook before selection/saving.


In [ ]:
checked_registry = registry.merge(audit_table, on="recording_id", how="left", validate="one_to_one")
assert len(checked_registry) == len(registry), "Recording count changed."
assert checked_registry[quality_columns + ["needs_timing_review"]].notna().all().all(), "Missing audit results."


## Step 21: Mark which recordings are eligible

**What:** require timing agreement, four quality passes, matching lengths and
at least two samples.
**Why:** a recording must not enter analysis simply because its timing flag is False.
`all(axis=1)` requires every quality check in that recording's row to be True.
This adds a flag; it does not delete rows or alter samples.


In [ ]:
checked_registry["eligible_for_analysis"] = (
    ~checked_registry["needs_timing_review"]
    & checked_registry[quality_columns].all(axis=1)
    & (checked_registry["sample_count"] == checked_registry["time_count"])
    & (checked_registry["sample_count"] >= 2)
)


## Step 22: Select the eligible development recordings

**What:** select records that belong to development and pass the eligibility rule.
**Why:** preserve the final-test partition for later evaluation.
The previous dataset retained 272 development recordings: K001 58, K002 75,
KA04 72 and KI04 67. The cell reports current results instead of forcing those counts.


In [ ]:
analysis_development = checked_registry[
    (checked_registry["partition"] == "development") & checked_registry["eligible_for_analysis"]
].copy()
print("Eligible development recordings:", len(analysis_development))
print(analysis_development.groupby(["bearing_id", "condition"]).size())


## Step 23: Check eligible coverage across operating settings

**What:** count eligible repetitions for every bearing and setting.
**Why:** exclusions are uneven. In the earlier run, K001 at N15_M07_F04 had
only 10 eligible repetitions. This affects the amount of evidence behind comparisons.


In [ ]:
display(analysis_development.groupby([
    "speed_code", "torq_code", "force_code", "bearing_id"
]).size().unstack("bearing_id", fill_value=0))


## Timing investigation: what we already learned

This reference note preserves the findings without adding another long sequence
of repeated file-loading cells. Values below are from our earlier real-data run;
the current audit tables above contain freshly calculated results.

| Recording | Observation |
|---|---|
| N15_M07_F10_K001_1 | 256,001 samples over about 4 s; full-span estimate 64,000.0205 Hz |
| N09_M07_F10_K001_1 | 256,823 samples over about 4 s; estimate 64,205.4451 Hz; nominal-grid endpoint difference about −12.84 ms |
| N15_M01_F10_K002_10 | 291,847 samples over about 4 s; estimate 72,961.5468 Hz |

For the last example, both phase-current channels and vibration shared the
unusual HostService sample count. Mechanical channels had 16,001 samples.
The declared measurement length and start/stop difference both indicated 4 s.
Origin metadata identified software/platform details, but did not explain the issue.

**Decision:** leave raw data and timestamps unchanged, hold failed recordings
from provisional analysis and keep the cause unresolved. Do not automatically
trim, resample, or set a new acquisition rate from the endpoint estimate.


## Step 24: Save the complete registry

**What:** save all recordings, including held cases and their eligibility flags.
**Why:** the registry is an audit trail; selecting 272 development rows must not
erase the other recordings. Three rate-related fields may be blank for unusable
timestamps; their timing-review flag must be True. Required fields stay complete.
This cell updates `metadata/recording_registry.csv`.


In [ ]:
timing_values = ["axis_span_s", "axis_rate_hz", "axis_rate_difference_pct"]
assert not checked_registry.drop(columns=timing_values).isna().any().any(), "Missing required results."
assert checked_registry.loc[~checked_registry["needs_timing_review"], timing_values].notna().all().all()
metadata_folder.mkdir(parents=True, exist_ok=True)
checked_registry.to_csv(metadata_folder / "recording_registry.csv", index=False)
print("Saved recordings:", len(checked_registry))


## Step 25: Save the two audit tables

**What:** also save timing and quality results as separate CSVs.
**Why:** keep the existing audit outputs available for investigation.
`index=False` avoids adding the pandas row index as a CSV column.


In [ ]:
timing_columns = ["recording_id", "sample_count", "time_count", "axis_span_s", "axis_rate_hz",
                  "axis_rate_difference_pct", "needs_timing_review", "timing_status"]
audit_table[timing_columns].to_csv(metadata_folder / "timing_audit.csv", index=False)
audit_table[["recording_id"] + quality_columns].to_csv(metadata_folder / "quality_audit.csv", index=False)


## Step 26: Read back the saved registry

**What:** read the CSV from disk and compare its recording IDs and eligibility flags.
**Why:** this is a real readback check, rather than displaying an in-memory copy.
With our 560-recording dataset, the registry contains 27 columns.


In [ ]:
saved_registry = pd.read_csv(metadata_folder / "recording_registry.csv")
assert saved_registry["recording_id"].tolist() == checked_registry["recording_id"].tolist()
assert saved_registry["eligible_for_analysis"].tolist() == checked_registry["eligible_for_analysis"].tolist()
print("Saved rows and columns:", saved_registry.shape)


## What this notebook has produced

- A fresh inventory built once from the files on disk.
- Fixed condition labels and bearing-separated partitions.
- Timing and quality checks, with retained and held cases visible.
- Three metadata CSVs, including `eligible_for_analysis` in the registry.

For Notebook 03 and later development work, select `partition == "development"`
**and** `eligible_for_analysis == True`. Preserve recording/bearing identities
when creating windows. Quality/timing exclusions may affect coverage and must
be reported. Retained recordings are still only repeated measurements of a few bearings.

The 64 kHz analysis grid remains provisional, physical amplitude units remain
unverified, and the cause of timing discrepancies remains unresolved.
No diagnostic feature selection has used final-test recordings here.
